# 00 - Setup and Environment

Establishes the runtime, verifies every dependency the pipeline needs, creates the
`Results/` tree, and records the exact software versions for the paper's
reproducibility table.

**Run this notebook first.** Every later notebook assumes `Experiment/` is importable
and that `Results/<dataset>/<stage>/` exists.

### Study design in one paragraph

The original submission benchmarked a multi-domain handcrafted feature pipeline
(1,002 features across time / frequency / time-frequency / spatial) with lightweight
classifiers, on a single DAS dataset, using a random stratified 80/10/10 split.
This revision keeps the methodology and changes three things: **two independent
datasets** instead of one, **group-aware splitting** that makes leakage structurally
impossible, and **feature selection re-fitted inside every cross-validation fold**
rather than once on the full training set.

In [1]:
import sys, platform, importlib, json, os
from pathlib import Path

EXPERIMENT_DIR = Path(r"/mnt/b6bdcd1c-136a-435b-aeed-0e1b31c32749/Paper_Reeyan/Experiment")
if str(EXPERIMENT_DIR) not in sys.path:
    sys.path.insert(0, str(EXPERIMENT_DIR))

import dasfe
from dasfe import config as C

print("python  :", platform.python_version())
print("platform:", platform.platform())
print("cpu cores:", os.cpu_count(), "-> n_jobs =", C.N_JOBS)
print("dasfe   :", dasfe.__version__)

python  : 3.13.9
platform: Linux-5.15.148-tegra-aarch64-with-glibc2.35
cpu cores: 12 -> n_jobs = 10
dasfe   : 1.0.0


## 1. Dependency check

Anything reported as MISSING here must be installed before continuing - the pipeline
has no fallbacks, because a silently-skipped selector would change the consensus
subset without warning.

In [2]:
REQUIRED = [
    "numpy", "scipy", "pandas", "sklearn", "matplotlib", "seaborn",
    "h5py", "pywt", "skimage", "joblib", "tqdm", "pyarrow",
    "lightgbm", "xgboost", "imblearn", "shap", "boruta", "skrebate", "mrmr",
]
OPTIONAL = ["torch"]

rows = []
for name in REQUIRED + OPTIONAL:
    try:
        m = importlib.import_module(name)
        rows.append((name, getattr(m, "__version__", "n/a"), "ok"))
    except ImportError as exc:
        rows.append((name, "-", f"MISSING ({exc.name})"))

import pandas as pd
env = pd.DataFrame(rows, columns=["package", "version", "status"])
missing = env[env.status.str.startswith("MISSING")]
display(env)
if len(missing):
    print("\nInstall the missing packages, e.g.:")
    print("  pip install " + " ".join(missing.package.replace(
        {"sklearn": "scikit-learn", "skimage": "scikit-image",
         "pywt": "PyWavelets", "imblearn": "imbalanced-learn",
         "mrmr": "mrmr-selection", "boruta": "Boruta"})))
else:
    print("\nAll required packages present.")

,package,version,status
0,numpy,2.3.5,ok
1,scipy,1.16.3,ok
2,pandas,2.3.3,ok
3,sklearn,1.7.2,ok
4,matplotlib,3.10.6,ok
5,seaborn,0.13.2,ok
6,h5py,3.15.1,ok
7,pywt,1.8.0,ok
8,skimage,0.25.2,ok
9,joblib,1.5.2,ok



Install the missing packages, e.g.:
  pip install lightgbm xgboost shap Boruta skrebate mrmr-selection torch


## 2. Dataset presence

We only check that the two dataset roots exist and are populated. No bulk data is
read here - notebook 01 builds the manifests.

In [3]:
print("Cao_2023    :", C.CAO_DIR, "->", "OK" if C.CAO_DIR.is_dir() else "NOT FOUND")
for split in ("Training", "Test"):
    d = C.CAO_DIR / split
    n = sum(1 for _ in d.rglob("*.mat")) if d.is_dir() else 0
    print(f"   {split:9s} {n:6d} .mat files")

print("\nTomasov_2024:", C.TOMASOV_DIR, "->", "OK" if C.TOMASOV_DIR.is_dir() else "NOT FOUND")
if C.TOMASOV_DIR.is_dir():
    total_gb = 0
    for cls in sorted(p for p in C.TOMASOV_DIR.iterdir() if p.is_dir()):
        h5 = list(cls.glob("*.h5"))
        gb = sum(p.stat().st_size for p in h5) / 1e9
        total_gb += gb
        print(f"   {cls.name:14s} {len(h5)} recording(s)  {gb:6.1f} GB")
    print(f"   {'TOTAL':14s} {total_gb:.1f} GB")

Cao_2023    : /mnt/b6bdcd1c-136a-435b-aeed-0e1b31c32749/Paper_Reeyan/Dataset/Cao_2023 -> OK
   Training   12335 .mat files
   Test        3084 .mat files

Tomasov_2024: /mnt/b6bdcd1c-136a-435b-aeed-0e1b31c32749/Paper_Reeyan/Dataset/Tomasov_2024/data -> OK
   car            4 recording(s)    12.6 GB
   construction   1 recording(s)     2.4 GB
   fence          3 recording(s)     3.7 GB
   longboard      2 recording(s)     5.5 GB
   manipulation   1 recording(s)     3.7 GB
   openclose      1 recording(s)     1.9 GB
   regular        1 recording(s)     5.7 GB
   running        1 recording(s)     2.7 GB
   walk           2 recording(s)     8.8 GB
   TOTAL          47.1 GB


## 3. Results tree

One directory per dataset, one sub-directory per pipeline stage. Every notebook
writes only into its own stage directory, so a stage can be deleted and re-run
without disturbing anything upstream or downstream.

In [4]:
for ds in ("cao", "tomasov"):
    for stage in C.STAGES:
        C.results_dir(ds, stage)
C.results_dir("shared", "figures")
C.results_dir("shared", "tables")

for p in sorted(C.RESULTS_DIR.rglob("*")):
    if p.is_dir():
        print(p.relative_to(C.RESULTS_DIR))

cao
cao/00_inventory
cao/01_splits
cao/02_windows
cao/03_features
cao/03_features_smoke
cao/03_features_smoke/shards
cao/03_features_smoke/shards/freq
cao/03_features_smoke/shards/ids
cao/03_features_smoke/shards/spatial
cao/03_features_smoke/shards/tf
cao/03_features_smoke/shards/time
cao/04_fusion
cao/04_fusion_smoke
cao/05_selection
cao/05_selection_smoke
cao/06_benchmark
cao/06_benchmark_smoke
cao/07_final
cao/07_final_smoke
cao/08_crossdataset
cao/09_report
shared
shared/08_crossdataset_smoke
shared/figures
shared/tables
tomasov
tomasov/00_inventory
tomasov/01_splits
tomasov/02_windows
tomasov/03_features
tomasov/03_features_smoke
tomasov/03_features_smoke/shards
tomasov/03_features_smoke/shards/freq
tomasov/03_features_smoke/shards/ids
tomasov/03_features_smoke/shards/spatial
tomasov/03_features_smoke/shards/tf
tomasov/03_features_smoke/shards/time
tomasov/04_fusion
tomasov/04_fusion_smoke
tomasov/05_selection
tomasov/05_selection_smoke
tomasov/06_benchmark
tomasov/06_benchmark_s

## 4. Feature-space contract

The four extractors expose a fixed dimensionality that is asserted at import time.
If any of these numbers drift, `import dasfe` fails immediately rather than
producing a silently different feature space.

In [5]:
import pandas as pd
counts = pd.DataFrame(
    [{"domain": d, "n_features": n} for d, n in dasfe.FEATURE_COUNTS.items()]
)
counts.loc[len(counts)] = ["MULTI-DOMAIN (fused)", dasfe.TOTAL_FEATURES]
display(counts)

print("window length :", C.WIN_LEN, "samples")
print("window hop    :", C.WIN_HOP, "samples  ->", 100 * (1 - C.WIN_HOP / C.WIN_LEN), "% overlap")
print("band-pass     :", C.BP_LOW, "-", C.BP_HIGH, "Hz, order", C.BP_ORDER)
print("STFT          : nfft", C.NFFT_STFT, "hop", C.HOP_STFT)
print("wavelet       :", C.WAVELET, "| DWT levels", C.DWT_LEVELS, "| WPT level", C.WPT_LEVEL)

,domain,n_features
0,time,112
1,freq,90
2,tf,739
3,spatial,61
4,MULTI-DOMAIN (fused),1002


window length : 8192 samples
window hop    : 2048 samples  -> 75.0 % overlap
band-pass     : 20.0 - 2000.0 Hz, order 4
STFT          : nfft 512 hop 128
wavelet       : db4 | DWT levels 6 | WPT level 4


### Why the window overlap matters

`WIN_HOP < WIN_LEN` means consecutive windows share 75% of their raw samples. That
is fine for *extraction* - it is how the annotation masks are laid out - but it makes
a random train/test split invalid, because a test window can share three quarters of
its samples with a training window. Notebook 02 handles this with guard bands.

In [6]:
import numpy as np, time
from dasfe import preprocess as pp, extract as ex

rng = np.random.default_rng(C.SEED)
patch = pp.preprocess_patch(rng.standard_normal((32, C.WIN_LEN)), C.TOMASOV.fs)
x1d = patch[16]

t0 = time.perf_counter()
feats = ex.extract_window(x1d, patch, C.TOMASOV.fs)
dt = time.perf_counter() - t0

for d, v in feats.items():
    print(f"  {d:8s} {v.shape[0]:4d} features   finite={np.isfinite(v).all()}")
print(f"\nall four domains, one window: {1000*dt:.1f} ms (single core)")

  time      112 features   finite=True
  freq       90 features   finite=True
  tf        739 features   finite=True
  spatial    61 features   finite=True

all four domains, one window: 174.3 ms (single core)


## 5. Environment record

Saved to `Results/shared/tables/environment.json` and reproduced verbatim in the
paper's specifications table.

In [7]:
record = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "cpu_count": os.cpu_count(),
    "n_jobs": C.N_JOBS,
    "seed": C.SEED,
    "dasfe_version": dasfe.__version__,
    "packages": {r[0]: r[1] for r in rows if r[2] == "ok"},
    "feature_counts": dasfe.FEATURE_COUNTS,
    "window": {"len": C.WIN_LEN, "hop": C.WIN_HOP,
               "bp_low": C.BP_LOW, "bp_high": C.BP_HIGH, "bp_order": C.BP_ORDER},
}
out = C.results_dir("shared", "tables") / "environment.json"
out.write_text(json.dumps(record, indent=2), encoding="utf-8")
print("wrote", out)

wrote /mnt/b6bdcd1c-136a-435b-aeed-0e1b31c32749/Paper_Reeyan/Results/shared/tables/environment.json


---
**Next:** `01_dataset_inventory_and_leakage_audit.ipynb` - build the manifests for both
datasets and quantify the leakage present in the published Cao train/test split.